# NB-12: WanVideoPipeline -- Full System Integration

## Learning Objectives
1. **SYS-01: Pipeline Composition** -- Understand how WanModel (DiT), WanVideoVAE, WanTextEncoder (T5/UMT5-XXL), and WanImageEncoder (CLIP ViT-H/14) compose into the full WanVideoPipeline that generates motion-controlled videos
2. **SYS-02: Tensor Data Flow** -- Trace any tensor from raw inputs (text prompt, reference image, control video) through T5/CLIP encoding, VAE latent encoding, 48-channel concatenation, DiT denoising, and VAE decoding to the output video `[B, 3, F, H, W]`
3. **SYS-03: Parameter Count Breakdown** -- Compute and interpret the parameter count across all four model components (DiT, VAE, T5, CLIP) and understand why the full pipeline requires ~8B parameters

## Prerequisites
- **NB-01**: RMSNorm, sinusoidal embeddings, and the `modulate` function -- these are the building blocks of timestep conditioning
- **NB-02**: QKV projections and multi-head attention layout -- the attention mechanism used in both T5 and CLIP
- **NB-03**: 3D Rotary Position Embeddings (RoPE) -- the spatial position encoding used in DiT
- **NB-04**: Self-attention and cross-attention in DiT blocks -- how text context is injected into the DiT
- **NB-05**: adaLN-Zero modulation -- how timestep conditioning gates each DiT block
- **NB-06**: DiT block -- the full attention + FFN block with adaLN gating
- **NB-07**: Patchify/unpatchify and Head -- how video frames become token sequences and back
- **NB-08**: WanModel end-to-end forward pass -- the full 48-channel DiT including gradient checkpointing
- **NB-09**: CausalConv3d, ResidualBlock, AttentionBlock -- the VAE building blocks
- **NB-10**: VAE Encoder3d -- downsampling video to latents
- **NB-11**: VAE Decoder3d and patchify disambiguation -- upsampling latents to video

## Concept Map
Each prior notebook covers one part of the pipeline:

| Notebook | Pipeline Stage |
|----------|---------------|
| NB-01 | RMSNorm + sinusoidal timestep embedding -- used by T5 (RMSNorm) and DiT (sinusoidal) |
| NB-02 | Attention primitives -- used by T5 text encoder and CLIP image encoder |
| NB-03 | 3D RoPE -- spatial position encoding inside DiT blocks |
| NB-04 | Cross-attention -- how T5 `context` and CLIP `clip_feature` condition the DiT |
| NB-05 | adaLN-Zero -- how FlowMatchScheduler timesteps gate each DiT block |
| NB-06 | DiT block -- the core unit repeated 30 times inside WanModel |
| NB-07 | Patchify/Head/unpatchify -- WanModel's input and output projections |
| NB-08 | WanModel forward pass -- the full DiT end-to-end |
| NB-09 | CausalConv3d + ResBlock + AttentionBlock -- VAE building blocks |
| NB-10 | VAE Encoder3d -- raw video `[B,3,F,H,W]` -> latents `[B,16,T_lat,H/8,W/8]` |
| NB-11 | VAE Decoder3d -- latents `[B,16,T_lat,H/8,W/8]` -> video `[B,3,F,H,W]` |
| **NB-12** | **Full pipeline** -- all components composed into WanVideoPipeline |

In [ ]:
# Source: established in NB-01/NB-08/NB-11, extended for all five pipeline modules
# Loads: wan_video_dit, wan_video_vae, wan_video_text_encoder, wan_video_image_encoder, flow_match
import sys
import types
import importlib.util
import pathlib

# ── Step 1: Find project root ─────────────────────────────────────────────────
# Walk up from notebook location (Course/) to find the directory containing diffsynth/
_here = pathlib.Path().resolve()
PROJECT_ROOT = None
_candidate = _here
for _ in range(10):  # search up to 10 levels
    if (_candidate / 'diffsynth' / 'models' / 'wan_video_dit.py').exists():
        PROJECT_ROOT = _candidate
        break
    _parent = _candidate.parent
    if _parent == _candidate:  # reached filesystem root
        break
    _candidate = _parent
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Could not find diffsynth/models/wan_video_dit.py. '
        'Run this notebook from Course/ inside the project checkout.'
    )
print(f'Project root: {PROJECT_ROOT}')

# ── Step 2: Stub modules that cause import errors ─────────────────────────────
# Stub wan_video_camera_controller (not needed for pipeline demos)
# Source: same pattern as NB-08 setup cell
_cam_stub = types.ModuleType('diffsynth.models.wan_video_camera_controller')
_cam_stub.SimpleAdapter = type('SimpleAdapter', (), {'__init__': lambda s, *a, **kw: None})
_diffsynth_stub = types.ModuleType('diffsynth')
_models_stub = types.ModuleType('diffsynth.models')
_diffusion_stub = types.ModuleType('diffsynth.diffusion')
sys.modules['diffsynth'] = _diffsynth_stub
sys.modules['diffsynth.models'] = _models_stub
sys.modules['diffsynth.models.wan_video_camera_controller'] = _cam_stub
sys.modules['diffsynth.diffusion'] = _diffusion_stub

# Note: tqdm is available in this environment -- do not stub it.
# The transformers library imports tqdm.auto which requires the real tqdm package.

# ── Step 3: Load modules in dependency order (wan_video_dit FIRST) ────────────
# wan_video_image_encoder.py imports flash_attention from wan_video_dit,
# so wan_video_dit must be registered in sys.modules before image_encoder loads.

# 3a. Load wan_video_dit.py
_dit_path = PROJECT_ROOT / 'diffsynth' / 'models' / 'wan_video_dit.py'
_spec = importlib.util.spec_from_file_location('diffsynth.models.wan_video_dit', _dit_path)
_dit_mod = importlib.util.module_from_spec(_spec)
sys.modules['diffsynth.models.wan_video_dit'] = _dit_mod
_spec.loader.exec_module(_dit_mod)

# 3b. Load wan_video_vae.py
_vae_path = PROJECT_ROOT / 'diffsynth' / 'models' / 'wan_video_vae.py'
_spec = importlib.util.spec_from_file_location('diffsynth.models.wan_video_vae', _vae_path)
_vae_mod = importlib.util.module_from_spec(_spec)
sys.modules['diffsynth.models.wan_video_vae'] = _vae_mod
_spec.loader.exec_module(_vae_mod)

# 3c. Load wan_video_text_encoder.py
_te_path = PROJECT_ROOT / 'diffsynth' / 'models' / 'wan_video_text_encoder.py'
_spec = importlib.util.spec_from_file_location('diffsynth.models.wan_video_text_encoder', _te_path)
_te_mod = importlib.util.module_from_spec(_spec)
sys.modules['diffsynth.models.wan_video_text_encoder'] = _te_mod
_spec.loader.exec_module(_te_mod)

# 3d. Load wan_video_image_encoder.py
_ie_path = PROJECT_ROOT / 'diffsynth' / 'models' / 'wan_video_image_encoder.py'
_spec = importlib.util.spec_from_file_location('diffsynth.models.wan_video_image_encoder', _ie_path)
_ie_mod = importlib.util.module_from_spec(_spec)
sys.modules['diffsynth.models.wan_video_image_encoder'] = _ie_mod
_spec.loader.exec_module(_ie_mod)

# 3e. Load flow_match.py
_fm_path = PROJECT_ROOT / 'diffsynth' / 'diffusion' / 'flow_match.py'
_spec = importlib.util.spec_from_file_location('diffsynth.diffusion.flow_match', _fm_path)
_fm_mod = importlib.util.module_from_spec(_spec)
sys.modules['diffsynth.diffusion.flow_match'] = _fm_mod
_spec.loader.exec_module(_fm_mod)

# ── Step 4: Import symbols ─────────────────────────────────────────────────────
from diffsynth.models.wan_video_dit import WanModel, sinusoidal_embedding_1d
from diffsynth.models.wan_video_vae import WanVideoVAE
from diffsynth.models.wan_video_text_encoder import WanTextEncoder
from diffsynth.models.wan_video_image_encoder import WanImageEncoder
from diffsynth.diffusion.flow_match import FlowMatchScheduler
import torch

print('Setup complete. Five modules loaded:')
print('  WanModel              <- diffsynth/models/wan_video_dit.py')
print('  WanVideoVAE           <- diffsynth/models/wan_video_vae.py')
print('  WanTextEncoder        <- diffsynth/models/wan_video_text_encoder.py')
print('  WanImageEncoder       <- diffsynth/models/wan_video_image_encoder.py')
print('  FlowMatchScheduler    <- diffsynth/diffusion/flow_match.py')

## Section 1: Pipeline Overview

The `WanVideoPipeline` is the orchestrator that composes four model components into a complete video generation system:

| Component | Class | Role |
|-----------|-------|------|
| DiT | `WanModel` (30 blocks) | Denoising backbone -- predicts velocity to remove noise |
| VAE | `WanVideoVAE` | Encodes/decodes video between pixel and latent space |
| T5 | `WanTextEncoder` (UMT5-XXL) | Encodes text prompts to semantic context vectors |
| CLIP | `WanImageEncoder` (ViT-H/14) | Encodes reference images to appearance feature tokens |

This is the capstone notebook that converges the **DiT track** (NB-01 to NB-08) and the **VAE track** (NB-09 to NB-11). Every concept from prior notebooks comes together here: sinusoidal timestep embeddings (NB-01), 3D RoPE (NB-03), cross-attention for text conditioning (NB-04), adaLN-Zero (NB-05), DiT blocks (NB-06), patchify/unpatchify (NB-07), the full DiT forward pass (NB-08), and the VAE encode/decode cycle (NB-10, NB-11).

**WanVideoPipeline unit architecture:** The pipeline orchestrates via five processing units (defined in `wan_video.py` lines 55-82):
- `PromptEmbedder` -- runs T5 and CLIP encoders to produce `context` and `clip_feature`
- `FunControl` -- VAE-encodes the control video into 16-channel latents
- `FunReference` -- VAE-encodes the reference image into 16-channel latents, then zero-pads to match temporal dimension
- `ImageEmbedderCLIP` -- combines control and reference latents as the `y` tensor (`[B, 32, T, H, W]`)
- `NoiseInitializer` -- generates the initial random noise latents `[B, 16, T, H, W]`

See `diffsynth/pipelines/wan_video.py` lines 55-82 for the full unit class definitions. Below we focus on the data flow and component-by-component behavior.

## Full Pipeline ASCII Diagram

The diagram below shows the complete data flow from raw inputs through all five pipeline stages to the output video. Every component box includes notebook back-references showing where each piece was taught.

```
Raw Inputs
    |
    +-- Text Prompt (str)
    |       |
    |       v
    |   WanTextEncoder (T5/UMT5-XXL, 24 layers)         [<- NB-01 RMSNorm, NB-02 Attention]
    |   vocab: 256,384  ->  dim: 4,096
    |   output: context [B, L<=512, 4096]
    |
    +-- Reference Image (PIL.Image)
    |       |
    |       +-->  WanImageEncoder (CLIP ViT-H/14, 32 layers)  [<- NB-02, NB-04]
    |       |    224x224  ->  257 tokens x 1,280 dim
    |       |    output: clip_feature [B, 257, 1280]
    |       |
    |       +-->  WanVideoVAE.encode (for reference latent)   [<- NB-10]
    |            [B, 3, 1, H, W]  ->  [B, 16, 1, H/8, W/8]
    |            output: ref_latents [B, 16, 1, H/8, W/8]
    |
    +-- Control Video (list of frames)
    |       |
    |       +-->  WanVideoVAE.encode                          [<- NB-10]
    |            [B, 3, F, H, W]  ->  [B, 16, T_lat, H/8, W/8]
    |            output: control_latents [B, 16, T_lat, H/8, W/8]
    |
    +-- Noise (Random Normal)
            [B, 16, T_lat, H/8, W/8]
            output: noise_latents [B, 16, T_lat, H/8, W/8]

                    +--------------------------------------------+
                    |   48-Channel Concatenation                 |  [<- NB-08]
                    |   x = noise_latents    [B, 16, T, H/8, W/8]|
                    |   y = cat(ctrl, ref)   [B, 32, T, H/8, W/8]|
                    |   -> [B, 48, T, H/8, W/8] inside WanModel  |
                    +--------------------+-----------------------+
                                         |
                    +--------------------v-----------------------+
                    |   WanModel (DiT, 30 blocks)                |  [<- NB-06 blocks,
                    |   patch_embedding: Conv3d(48,1536,k=1,2,2) |      NB-07 patchify,
                    |   Patchify -> [B, T*H/2*W/2, 1536] tokens  |      NB-08 forward]
                    |                                            |
                    |   Inputs also injected:                    |
                    |   +-- context [B, L, 4096] -> [B, L, 1536] |  [<- NB-04 cross-attn]
                    |   +-- clip_feature -> [B, 257, 1536]        |  [<- NB-02]
                    |   |   prepended to context sequence         |
                    |   +-- t_mod [B, 6, 1536]   (adaLN-Zero)    |  [<- NB-05]
                    |   +-- freqs  (3D RoPE)                      |  [<- NB-03]
                    |                                            |
                    |   x 50 timesteps via FlowMatchScheduler    |
                    |   (velocity prediction + scheduler.step)   |
                    |   FlowMatchScheduler                        |  [<- new in NB-12]
                    +--------------------+-----------------------+
                                         |
                              denoised latents
                              [B, 16, T_lat, H/8, W/8]
                                         |
                    +--------------------v-----------------------+
                    |   WanVideoVAE.decode                        |  [<- NB-11]
                    |   Decoder3d upsamples x8 spatial, x4 temp   |
                    |   T_lat = (F-1)//4 + 1 (e.g. 21 for 81fr)  |
                    +--------------------+-----------------------+
                                         |
                               Output Video
                               [B, 3, F, H, W]
```

**Temporal compression note:** The VAE compresses `F` frames to `T_lat = (F-1)//4 + 1` latent frames.
Examples: 5 frames -> 2 latent frames, 17 frames -> 5, 81 frames -> 21.
Use `(F-1)//4 + 1`, NOT `F//4` -- the latter gives wrong counts (see NB-11 disambiguation).

## Section 2: Component Demos

The next four cells each instantiate one pipeline component with a **reduced configuration** for fast CPU execution (per STD-03: each cell must run in under 5 seconds). The demo configs use tiny vocabularies, fewer layers, or small input tensors while keeping production architecture dimensions.

Each demo:
1. Instantiates the component with reduced config (or full config if it's already fast)
2. Runs a forward pass with dummy inputs
3. Asserts the expected output shape
4. Prints a parameter count comparison (demo vs production)

The four demo cells are self-contained -- you can run them independently after the setup cell.

In [ ]:
# Source: diffsynth/models/wan_video_text_encoder.py, class WanTextEncoder line 212
# DEMO CONFIG: vocab=1000, num_layers=2 (production: vocab=256,384, num_layers=24)
# CRITICAL: Full WanTextEncoder() takes ~35s to instantiate on CPU due to
#   Embedding(256384, 4096) = 1.05B parameters initialized at construction time.

text_encoder = WanTextEncoder(
    vocab=1000,      # reduced from 256,384 for speed
    dim=4096,        # production architecture dimension
    dim_attn=4096,
    dim_ffn=10240,
    num_heads=64,
    num_layers=2,    # reduced from 24 for speed
    num_buckets=32,
)

dummy_ids  = torch.zeros(1, 10, dtype=torch.long)   # [B, L]
dummy_mask = torch.ones(1, 10)                       # [B, L]
with torch.no_grad():
    context = text_encoder(dummy_ids, dummy_mask)    # -> [B, L, 4096]

assert context.shape == (1, 10, 4096), f'Expected (1, 10, 4096), got {context.shape}'
print(f'T5 output shape: {context.shape}')
print(f'  Production: vocab=256,384, num_layers=24 -> ~5.68B params')
print(f'  Demo:       vocab=1,000,  num_layers=2  -> {sum(p.numel() for p in text_encoder.parameters()):,} params')
print()
print('Shape: [B=1, L=10, dim=4096] -- each of 10 tokens becomes a 4096-dim context vector')
print('DiT injects context via cross-attention (NB-04) after projecting 4096->1536')

In [ ]:
# Source: diffsynth/models/wan_video_image_encoder.py, class WanImageEncoder line ~852
# Full CLIP ViT-H/14 -- no reduced config needed (2.4s init + ~0.5s forward < 5s)
# WanImageEncoder wraps clip_xlm_roberta_vit_h_14 with pretrained=False for fast init

image_encoder = WanImageEncoder()

dummy_img = torch.randn(1, 3, 224, 224)                           # [B, 3, 224, 224]
with torch.no_grad():
    clip_feature = image_encoder.encode_image([dummy_img])         # list input -> [B, 257, 1280]

assert clip_feature.shape == (1, 257, 1280), f'Expected (1, 257, 1280), got {clip_feature.shape}'
print(f'CLIP output shape: {clip_feature.shape}')
print(f'  257 tokens = 1 [CLS] token + 256 patch tokens  (224/14 = 16, 16*16 = 256)')
print(f'  Parameters: {sum(p.numel() for p in image_encoder.parameters()):,}')
print()
print('Shape: [B=1, 257 tokens, 1280-dim] -- appearance features from the reference image')
print('DiT prepends clip_feature tokens to the text context sequence (after dim projection 1280->1536)')

In [ ]:
# Source: diffsynth/models/wan_video_vae.py, class WanVideoVAE line 1058
# Full VAE with tiny spatial input -- fast on CPU
# WanVideoVAE wraps VideoVAE_ (1.3B param VAE) with learned mean/std normalization

vae = WanVideoVAE()

dummy_video = torch.randn(1, 3, 5, 32, 32)   # [B, C=3, F=5, H=32, W=32] -- tiny input
with torch.no_grad():
    latents = vae.encode(dummy_video, device='cpu')   # [B, 16, T_lat, H/8, W/8]
    decoded = vae.decode(latents, device='cpu')        # [B, 3, F, H, W]

# Temporal compression: T_lat = (F-1)//4 + 1 = (5-1)//4 + 1 = 2
# Spatial compression: H/8 = 32/8 = 4,  W/8 = 32/8 = 4
assert latents.shape == (1, 16, 2, 4, 4), f'Expected (1, 16, 2, 4, 4), got {latents.shape}'
assert decoded.shape == dummy_video.shape, f'Expected {dummy_video.shape}, got {decoded.shape}'

print(f'VAE encode: {dummy_video.shape} -> {latents.shape}')
print(f'VAE decode: {latents.shape} -> {decoded.shape}')
print(f'  Spatial:  /8  ({dummy_video.shape[3]} -> {dummy_video.shape[3]//8})')
print(f'  Temporal: (F-1)//4+1 ({dummy_video.shape[2]} -> {latents.shape[2]})')
print(f'  Latent channels: 3 -> 16')
print(f'  Parameters: {sum(p.numel() for p in vae.parameters()):,}')

### DiT Forward Pass Demo

The DiT (Diffusion Transformer) is the heart of the denoising process. Readers who completed NB-08 have already seen the full 48-channel `WanModel` forward pass. Here we use a base T2V (text-to-video) configuration with `in_dim=16` and `num_layers=3` (vs production 30), matching the same reduced-config approach.

**Key difference from NB-08:** In the full Fun-Control pipeline the model uses `in_dim=48` (48 = 16 noise + 16 ctrl + 16 ref). In this demo we use `in_dim=16` with `has_image_input=False` to demonstrate the base DiT forward pass cleanly. This is the same configuration used in base Wan2.1-T2V.

**Source:** `diffsynth/models/wan_video_dit.py`, `WanModel.__init__` line 274

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, WanModel.__init__ line 274
# DEMO CONFIG: num_layers=3 (production: 30), in_dim=16 (base T2V config)
# Verified: 0.042s on CPU for this config (STD-03 compliant)

dit = WanModel(
    dim=1536,            # production architecture dimension
    in_dim=16,           # base T2V config (production Fun-Control uses in_dim=48)
    ffn_dim=8960,        # production FFN dimension
    out_dim=16,          # output: 16 latent channels
    text_dim=4096,       # T5 text encoder output dim
    freq_dim=256,        # sinusoidal timestep embedding dim
    eps=1e-6,
    patch_size=(1, 2, 2),
    num_heads=12,
    num_layers=3,        # REDUCED from 30 for CPU demo
    has_image_input=False,  # base T2V config -- no CLIP image path
)

# Dummy inputs matching DiT contract
B, F, H, W = 1, 4, 8, 8               # small spatial dims for CPU demo
x = torch.randn(B, 16, F, H, W)        # noise latents [B, in_dim, F, H, W]
timestep = torch.tensor([500.0])        # single timestep value [B]
ctx = torch.randn(B, 10, 4096)          # text encoder output [B, L, 4096]

with torch.no_grad():
    dit.eval()
    out = dit(x, timestep, ctx)          # -> [B, out_dim, F, H, W]

assert out.shape == torch.Size([B, 16, F, H, W]), f'Expected {(B, 16, F, H, W)}, got {out.shape}'
print(f'DiT input:  {x.shape}     (noise latents)')
print(f'DiT output: {out.shape}   (velocity prediction -- direction to clean image)')
print(f'  Demo (3 layers, in_dim=16):   {sum(p.numel() for p in dit.parameters()):,} params')
print(f'  Production (30 layers, in_dim=16): ~1.42B params (base T2V)')
print(f'  Production (30 layers, in_dim=48): ~1.56B params (Fun-Control)')

## How Component Outputs Connect

Each component's output feeds directly into the next stage. Referring back to the pipeline diagram in Cell 3:

| Producer | Output | Consumer | How it enters |
|----------|--------|----------|---------------|
| `WanTextEncoder` | `context [B, L, 4096]` | `WanModel` | `text_embedding` MLP projects 4096 -> 1536; consumed by cross-attention in each DiT block (NB-04) |
| `WanImageEncoder` | `clip_feature [B, 257, 1280]` | `WanModel` | `img_emb` MLP projects 1280 -> 1536; **prepended** to context sequence before DiT attention |
| `WanVideoVAE.encode` | `control_latents [B, 16, T, H/8, W/8]` | `WanModel.forward` | Combined with ref_latents as `y=[B,32,T,H/8,W/8]`; WanModel concatenates `x` and `y` internally to form the 48-channel input |
| `WanVideoVAE.encode` | `ref_latents [B, 16, 1, H/8, W/8]` | `WanModel.forward` | Same as above |
| `WanModel` | `velocity [B, 16, T, H/8, W/8]` | `FlowMatchScheduler.step` | `prev = latents + velocity * (sigma_next - sigma_current)` -- since sigma decreases each step, delta is negative, moving latents toward clean image |
| `FlowMatchScheduler` | updated `latents [B, 16, T, H/8, W/8]` | (repeated 50 times) | After 50 steps, latents are denoised |
| `WanVideoVAE.decode` | `output_video [B, 3, F, H, W]` | (final output) | Decoder3d upsamples x8 spatial, x4 temporal |

**Anti-pattern reminder:** Do NOT pass a pre-concatenated 48-channel tensor to `WanModel.forward`. The method signature is:
```python
def forward(self, x, timestep, context, clip_feature=None, y=None, ...):
```
where `x=[B,16,T,H,W]` (noise latents) and `y=[B,32,T,H,W]` (control+ref latents). WanModel concatenates them internally when `has_image_input=True`.

Source: `diffsynth/models/wan_video_dit.py`, WanModel.forward line 359